# Phase 1 · 04 — MCP **with the library** (SOTA)

In `03` you implemented MCP's JSON-RPC by hand. Now let the `mcp` library do the
plumbing so you only write tools. Our server lives in `mcp_server.py` and exposes
five tools: `add`, `subtract`, `multiply`, `divide`, `power`.

Two things the library gives you for free vs. `03`:
- **Schema generation** — the tool's JSON schema comes from the function's type
  hints (you hand-wrote those schemas in `mcp_server_raw.py`).
- **The handshake, validation, and transports** — no manual `initialize`.

> We use **mcp 2.x** (`MCPServer`; this was called `FastMCP` in 1.x).

## 1. Connect a client and discover the tools

`mcp.Client` can talk to a server object **in-process** — ideal for a notebook,
no subprocess needed. MCP is async, so we use `async def` + `await`
(Jupyter supports top-level `await`).

In [ ]:
# %pip install "mcp[cli]"
from mcp import Client
from mcp_server import mcp as math_server   # the MCPServer object from mcp_server.py

async def list_the_tools():
    async with Client(math_server) as session:
        return (await session.list_tools()).tools

tools = await list_the_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

## 2. Call a tool

We didn't hand-write these schemas — the library generated them from the
functions' type hints. A call returns content blocks; the text is in
`.content[0].text`, and a typed value is in `.structured_content`.

In [ ]:
async def call_tool(name, arguments):
    async with Client(math_server) as session:
        return await session.call_tool(name, arguments)

res = await call_tool("power", {"base": 2, "exponent": 10})
print("text result:      ", res.content[0].text)
print("structured result:", res.structured_content)

## 3. The auto-generated schema

Compare this to the schema you wrote by hand in `mcp_server_raw.py` — same shape,
but here it was derived from `def add(a: float, b: float)`.

In [ ]:
import json
add_tool = next(t for t in tools if t.name == "add")
print(json.dumps(add_tool.input_schema, indent=2))

## 4. Bridge MCP → the LLM

This ties the whole phase together. We (1) ask the MCP server what tools exist,
(2) convert each to the OpenAI tool schema, and (3) run the **same tool-calling
loop from `02`** — except every tool call is executed **by the MCP server**.

Add a tool to `mcp_server.py` and it shows up here automatically.

In [ ]:
import os, json
from openai import OpenAI

MODEL = "gpt-4o-mini"
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY not set — set it and restart the kernel to run this cell.")

def to_openai_schema(mcp_tool):
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description,
            "parameters": mcp_tool.input_schema,
        },
    }

async def run_with_mcp(user_message):
    llm = OpenAI()                          # needs OPENAI_API_KEY
    async with Client(math_server) as session:
        openai_tools = [to_openai_schema(t) for t in (await session.list_tools()).tools]
        messages = [{"role": "user", "content": user_message}]
        while True:
            response = llm.chat.completions.create(
                model=MODEL, messages=messages, tools=openai_tools, tool_choice="auto",
            )
            msg = response.choices[0].message
            messages.append(msg)
            if not msg.tool_calls:
                return msg.content
            for call in msg.tool_calls:
                args = json.loads(call.function.arguments)
                value = (await session.call_tool(call.function.name, args)).content[0].text
                print(f"  🔧 MCP {call.function.name}({args}) -> {value}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": value})

answer = await run_with_mcp("Take 7 to the power of 3, divide it by 2, then subtract 10.")
print("\nFinal answer:", answer)

## 5. Running it as a real stdio server

In-process is convenient for learning. In production the server is a separate
process launched over stdio (just like your subprocess demo in `03`).
`mcp_server.py` already supports it — the last line is `mcp.run(transport="stdio")`:

```bash
python mcp_server.py
```

To connect a client to that separate process instead of in-process, pass
`StdioServerParameters(command="python", args=["mcp_server.py"])` to `Client(...)`.
Same `list_tools()` / `call_tool()` API — only the transport changes.

## Phase 1 complete 🎉

| Step | From scratch | With the library |
|---|---|---|
| Tool calling | `01` — protocol + parser + loop by hand | `02` — OpenAI structured tool calls |
| MCP | `03` — JSON-RPC over stdio by hand | `04` — the `mcp` library (this notebook) |

You now understand both *what* these systems do and *how the SOTA tools package
it*. Adding a capability is a one-function edit in `mcp_server.py`.